## Defualt params:
`(500, 500) and (1000, 1000) network area are *NOT* tested for node placement.`
1. Initial energy= 0.5J
2. Comm range= 50m
3. Sink speed= 25m/round
4. Net area= (100, 100)
5. Nodes num= 100

In [ ]:
import sys
sys.path.append(
    r"D:\Papers\4) Finished Articles\6. MWSN - DCHPC\DCHPC\ModelClasses")

from ConfigClass.config import plot_comparison, visualize_deployment
from simulation import Simulation

## Basic Methods

In [ ]:
def plot_time_series(time_series_dict, label, title, color='b'):
    rounds_list = time_series_dict['rounds']
    values_list = time_series_dict[label]

    min_len = min(len(r) for r in rounds_list)

    round_grid = np.full((len(rounds_list), min_len), np.nan)
    value_grid = np.full((len(values_list), min_len), np.nan)

    for i, (rds, vals) in enumerate(zip(rounds_list, values_list)):
        round_grid[i, :] = rds[:min_len]
        value_grid[i, :] = vals[:min_len]

    avg_rounds = np.nanmean(round_grid, axis=0)
    avg_vals = np.nanmean(value_grid, axis=0)
    std_vals = np.nanstd(value_grid, axis=0)

    valid = ~np.isnan(avg_vals)
    avg_rounds = avg_rounds[valid]
    avg_vals = avg_vals[valid]
    std_vals = std_vals[valid]

    plt.plot(avg_rounds, avg_vals,
             label=f'{title} (Mean)', color=color, linewidth=2)
    plt.fill_between(avg_rounds,
                     avg_vals - std_vals,
                     avg_vals + std_vals,
                     color=color, alpha=0.2, label=f'{title} ± Std')

In [ ]:
import numpy as np
import random


def run_multiple_simulations(area_size, n_nodes,
                             sink_mode, routing_mode, mode,
                             n_runs=30, CHS="optimizer",
                             edge_threshold=0.4, tune_edge_iterations=20,
                             include_ack_energy=False,
                             num_sinks=1,
                             enable_heterogeneity=False, hetero_mode='two_tier',
                             variable_packet_size=False):

    FND, HND, LND, PDR = [], [], [], []
    TotalGenerated, TotalDelivered = [], []
    Avg_E2E_Delay_Rounds, Avg_E2E_Delay_Sec = [], []

    EC, avg_RE, TH, TH_pps, PLR, LB, FI, CE, Buffer_Overflow_Rate = [
    ], [], [], [], [], [], [], [], []
    CA, RL, EE, EE_Js, routing_Overhead_Bytes, Traffic_Load_Pct, Overhead_Normalized = [
    ], [], [], [], [], [], []

    round_runs, EC_runs, avg_RE_runs = [], [], []
    TH_runs, TH_pps_runs, PLR_runs, LB_runs = [], [], [], []
    FI_runs, CE_runs, Buffer_Overflow_Rate_runs = [], [], []
    CA_runs, RL_runs, EE_runs, EE_Js_runs = [], [], [], []
    routing_Overhead_Bytes_runs, Traffic_Load_Pct_runs, Overhead_Normalized_runs = [], [], []

    for seed in range(n_runs):
        np.random.seed(seed)
        random.seed(seed)
        sim = Simulation(
            area_size=area_size,
            n_nodes=n_nodes,
            rounds=60000,
            init_energy=0.5,
            comm_range=50.0,
            sink_mode=sink_mode,
            routing_mode=routing_mode,
            seed=seed,
            localization_mode=mode,
            head_selection_strategy=CHS,

            edge_threshold=edge_threshold,
            tune_edge_iterations=tune_edge_iterations,

            include_ack_energy=include_ack_energy,
            num_sinks=num_sinks,

            enable_heterogeneity=enable_heterogeneity,
            hetero_mode=hetero_mode,
            variable_packet_size=variable_packet_size
        )
        metrics, detailed_metrics = sim.run()

        rounds = np.array(detailed_metrics['round'])
        EC_vals = np.array(detailed_metrics['EC'])
        avg_RE = np.array(detailed_metrics['avg_RE'])
        TH = np.array(detailed_metrics['TH'])
        TH_pps = np.array(detailed_metrics['TH_pps'])
        PLR = np.array(detailed_metrics['PLR'])
        LB = np.array(detailed_metrics['LB'])
        FI = np.array(detailed_metrics['FI'])
        CE = np.array(detailed_metrics['CE'])
        Buffer_Overflow_Rate = np.array(
            detailed_metrics['Buffer_Overflow_Rate'])
        CA = np.array(detailed_metrics['CA'])
        RL = np.array(detailed_metrics['RL'])
        EE = np.array(detailed_metrics['EE'])
        EE_Js = np.array(detailed_metrics['EE_Js'])
        routing_Overhead_Bytes = np.array(
            detailed_metrics['Routing_Overhead_Bytes'])
        Traffic_Load_Pct = np.array(detailed_metrics['Traffic_Load_Pct'])
        Overhead_Normalized = np.array(detailed_metrics['Overhead_Normalized'])

        round_runs.append(rounds)
        EC_runs.append(EC_vals)
        avg_RE_runs.append(avg_RE)
        TH_runs.append(TH)
        TH_pps_runs.append(TH_pps)
        PLR_runs.append(PLR)
        LB_runs.append(LB)
        FI_runs.append(FI)
        CE_runs.append(CE)
        Buffer_Overflow_Rate_runs.append(
            Buffer_Overflow_Rate)
        CA_runs.append(CA)
        RL_runs.append(RL)
        EE_runs.append(EE)
        EE_Js_runs.append(EE_Js)
        routing_Overhead_Bytes_runs.append(
            routing_Overhead_Bytes)
        Traffic_Load_Pct_runs.append(Traffic_Load_Pct)
        Overhead_Normalized_runs.append(Overhead_Normalized)

        plot_res = {
            'rounds': round_runs,
            'EC': EC_runs,
            'avg_RE': avg_RE_runs,
            'TH': TH_runs,
            'TH_pps': TH_pps_runs,
            'PLR': PLR_runs,
            'LB': LB_runs,
            'FI': FI_runs,
            'CE': CE_runs,
            'Buffer_Overflow_Rate': Buffer_Overflow_Rate_runs,
            'CA': CA_runs,
            'RL': RL_runs,
            'EE': EE_runs,
            'EE_Js': EE_Js_runs,
            'routing_Overhead_Bytes': routing_Overhead_Bytes_runs,
            'Traffic_Load_Pct': Traffic_Load_Pct_runs,
            'Overhead_Normalized': Overhead_Normalized_runs

        }

        FND.append(metrics['FND'])
        HND.append(metrics['HND'])
        LND.append(metrics['LND'])
        PDR.append(metrics['PDR'])
        TotalGenerated.append(metrics['TotalGenerated'])
        TotalDelivered.append(metrics['TotalDelivered'])
        Avg_E2E_Delay_Rounds.append(metrics['Avg_E2E_Delay_Rounds'])
        Avg_E2E_Delay_Sec.append(metrics['Avg_E2E_Delay_Sec'])
        print(f"Seed = {seed}, Mode = {mode} results:")
        print(
            f"FND: {metrics['FND']}, HND: {metrics['HND']}, LND: {metrics['LND']}")
        print(
            f"PDR: {metrics['PDR']}, Total Generated: {metrics['TotalGenerated']}, Total Delivered: {metrics['TotalDelivered']}")
        print(
            f"Avg E2E Delay Rounds: {metrics['Avg_E2E_Delay_Rounds']}, Avg E2E Delay Sec: {metrics['Avg_E2E_Delay_Sec']}")
        print("\n")

    return [np.array(FND), np.array(HND), np.array(LND),
            np.array(PDR), np.array(TotalGenerated), np.array(TotalDelivered),
            np.array(Avg_E2E_Delay_Rounds), np.array(Avg_E2E_Delay_Sec)], plot_res

## Reporting variance, confidence bounds, or statistical significance 

In [ ]:
import os
import sys
import json
import math
import random
from pathlib import Path
from typing import Dict, Any, List, Tuple
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests

# Directory Setup
RESULTS_DIR = Path("results")
PLOTS_DIR = Path("plots")
RESULTS_DIR.mkdir(exist_ok=True)
PLOTS_DIR.mkdir(exist_ok=True)

# BASELINE CONFIGURATION (RUN ONCE)
BASELINE_CONFIG = {
    "area_size": (100, 100),
    "n_nodes": 100,
    "rounds": 60000,
    "init_energy": 0.5,
    "comm_range": 50.0,
    "sink_mode": "eeosp",
    "routing_mode": "multi-hop",
    "head_selection_strategy": "optimizer",
    "edge_threshold": 0.4,
    "tune_edge_iterations": 20,
    "include_ack_energy": False,
    "num_sinks": 1,
    "enable_heterogeneity": False,
    "hetero_mode": "two_tier",
    "variable_packet_size": False,
    "sink_policy": "nearest",
}

# SCENARIO DEFINITIONS
SCENARIOS = {
    "num_sinks": {
        "description": "Effect of sink count (baseline: 1 sink)",
        "varying_param": "num_sinks",
        "configs": [
            {"name": "2_sinks", "num_sinks": 2},
            {"name": "3_sinks", "num_sinks": 3},
        ]
    },
    "sink_mobility": {
        "description": "Sink mobility strategies (baseline: eeosp)",
        "varying_param": "sink_mode",
        "configs": [
            {"name": "fixed", "sink_mode": "fixed"},
            {"name": "random", "sink_mode": "random"},
            {"name": "adaptive", "sink_mode": "adaptive"},
        ]
    },
    "routing": {
        "description": "Routing strategies (baseline: multi-hop)",
        "varying_param": "routing_mode",
        "configs": [
            {"name": "single_hop", "routing_mode": "single-hop"},
        ]
    },
    "heterogeneity": {
        "description": "Node energy heterogeneity (baseline: homogeneous)",
        "varying_param": "enable_heterogeneity",
        "configs": [
            {"name": "weak_hetero", "enable_heterogeneity": True, "hetero_mode": "weak"},
            {"name": "two_tier_hetero", "enable_heterogeneity": True,
                "hetero_mode": "two_tier"},
        ]
    },
    "packet_size": {
        "description": "Packet size variation (baseline: fixed)",
        "varying_param": "variable_packet_size",
        "configs": [
            {"name": "variable_size", "variable_packet_size": True},
        ]
    },
    "ack_energy": {
        "description": "ACK energy modeling (baseline: excluded)",
        "varying_param": "include_ack_energy",
        "configs": [
            {"name": "with_ack", "include_ack_energy": True},
        ]
    },
    "ch_selection": {
        "description": "CH selection strategies (baseline: optimizer)",
        "varying_param": "head_selection_strategy",
        "configs": [
            {"name": "random_ch", "head_selection_strategy": "random"},
            {"name": "adaptive_ch", "head_selection_strategy": "adaptive"},
        ]
    },
    "sink_policy": {
        "description": "Sink assignment policies (requires 2 sinks)",
        "varying_param": "sink_policy",
        "configs": [
            {"name": "nearest_2sinks", "sink_policy": "nearest", "num_sinks": 2},
            {"name": "load_aware_2sinks", "sink_policy": "load_aware", "num_sinks": 2},
            {"name": "balanced_2sinks", "sink_policy": "balanced", "num_sinks": 2},
        ],
        "requires_custom_baseline": True,
        "custom_baseline_config": {"num_sinks": 2, "sink_policy": "nearest"}
    }
}

# METRICS
SCALAR_METRICS = [
    "FND", "HND", "LND", "PDR", "TotalDelivered", "TotalGenerated",
    "Avg_E2E_Delay_Rounds", "Avg_E2E_Delay_Sec", "RoutingOverhead"
]

# ONLY collect guaranteed numeric time-series (skip RL_per_sink dicts)
TS_METRICS = [
    "round", "EC", "avg_RE", "TH", "TH_pps", "PDR", "PLR", "LB", "FI", "CA",
    "RL", "RL_primary", "RL_nearest", "RL_assigned", "EE", "EE_Js", "CE",
    "E2E_Delay_Rounds", "E2E_Delay_Sec", "Buffer_Overflow_Rate",
    "Routing_Overhead_Bytes", "Traffic_Load_Pct", "Overhead_Normalized"
]

# STATISTICAL HELPERS


def ci95_t(data: np.ndarray) -> Tuple[float, float, float]:
    """Exact t-distribution 95% CI."""
    arr = np.asarray(data, dtype=float)
    arr = arr[np.isfinite(arr)]
    n = len(arr)
    if n < 2:
        return (np.nan, np.nan, np.nan)
    mean = np.mean(arr)
    se = np.std(arr, ddof=1) / math.sqrt(n)
    tcrit = stats.t.ppf(0.975, df=n-1)
    return (mean, mean - se*tcrit, mean + se*tcrit)


def paired_cohens_d(x: np.ndarray, y: np.ndarray) -> float:
    """Paired Cohen's d."""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    diff = x[mask] - y[mask]
    if len(diff) <= 1 or np.std(diff, ddof=1) == 0:
        return 0.0
    return float(np.mean(diff) / (np.std(diff, ddof=1) + 1e-12))


def run_simulation(seed: int, config: Dict[str, Any]) -> Tuple[Dict[str, float], Dict[str, List[Any]]]:
    """Run single simulation with robust metric collection."""
    np.random.seed(seed)
    random.seed(seed)

    cfg = BASELINE_CONFIG.copy()
    # Exclude 'name' key
    cfg.update({k: v for k, v in config.items() if k != 'name'})
    cfg['seed'] = seed

    sim = Simulation(**cfg)
    metrics, detailed = sim.run()

    # Filter detailed metrics to ONLY numeric time-series (skip dicts like RL_per_sink)
    clean_detailed = {}
    for k, v in detailed.items():
        if k not in TS_METRICS:
            continue
        # Skip if value is a dict or non-numeric list
        if isinstance(v, dict):
            continue
        if isinstance(v, list) and v and not isinstance(v[0], (int, float, np.number)):
            continue
        clean_detailed[k] = v

    return metrics, clean_detailed

# CORE EVALUATION ENGINE


class EfficientEvaluator:
    def __init__(self, seeds: int = 30):
        self.seeds = seeds
        self.baseline_scalars = None
        self.baseline_ts = None
        self.custom_baselines = {}

    def run_baseline_once(self):

        scalars = {m: [] for m in SCALAR_METRICS}
        ts = {m: [] for m in TS_METRICS}

        for seed_idx, seed in enumerate(self.seeds):
            print(f"  Baseline seed {seed_idx+1}/{self.seeds}", end='\r')
            metrics, detailed = run_simulation(seed, {})

            for m in SCALAR_METRICS:
                scalars[m].append(metrics.get(m, np.nan))

            for m in TS_METRICS:
                if m in detailed and isinstance(detailed[m], list) and len(detailed[m]) > 0:
                    # Convert safely to float array
                    try:
                        arr = np.array(detailed[m], dtype=float)
                        ts[m].append(arr)
                    except:
                        ts[m].append(np.array([]))
                else:
                    ts[m].append(np.array([]))
        print()

        for m in SCALAR_METRICS:
            scalars[m] = np.array(scalars[m], dtype=float)

        self.baseline_scalars = scalars
        self.baseline_ts = ts
        print("Baseline completed and cached\n")
        return scalars, ts

    def run_custom_baseline(self, scenario_name: str, custom_cfg: Dict[str, Any]):
        """Run special baseline for scenarios needing it."""
        print(f"Running custom baseline for {scenario_name}...")

        scalars = {m: [] for m in SCALAR_METRICS}
        ts = {m: [] for m in TS_METRICS}

        for seed in self.seeds:
            metrics, detailed = run_simulation(seed, custom_cfg)

            for m in SCALAR_METRICS:
                scalars[m].append(metrics.get(m, np.nan))

            for m in TS_METRICS:
                if m in detailed and isinstance(detailed[m], list) and len(detailed[m]) > 0:
                    try:
                        arr = np.array(detailed[m], dtype=float)
                        ts[m].append(arr)
                    except:
                        ts[m].append(np.array([]))
                else:
                    ts[m].append(np.array([]))

        for m in SCALAR_METRICS:
            scalars[m] = np.array(scalars[m], dtype=float)

        self.custom_baselines[scenario_name] = (scalars, ts)
        return scalars, ts

    def run_scenario(self, scenario_name: str, scenario_def: Dict[str, Any]):
        """Run scenario configs against cached baseline with robust stats."""
        print(f"\n{'='*70}")
        print(f"SCENARIO: {scenario_name}")
        print(f"Description: {scenario_def['description']}")
        print(f"Varying: {scenario_def['varying_param']}")
        print(f"{'='*70}")

        # Get appropriate baseline
        if scenario_def.get("requires_custom_baseline", False):
            if scenario_name not in self.custom_baselines:
                custom_cfg = scenario_def.get("custom_baseline_config", {})
                baseline_scalars, baseline_ts = self.run_custom_baseline(
                    scenario_name, custom_cfg)
            else:
                baseline_scalars, baseline_ts = self.custom_baselines[scenario_name]
            baseline_name = f"baseline_{scenario_name}"
        else:
            if self.baseline_scalars is None:
                self.run_baseline_once()
            baseline_scalars = self.baseline_scalars
            baseline_ts = self.baseline_ts
            baseline_name = "baseline"

        results_dir = RESULTS_DIR / scenario_name
        plots_dir = PLOTS_DIR / scenario_name
        results_dir.mkdir(exist_ok=True)
        plots_dir.mkdir(exist_ok=True)

        scenario_results = {}
        all_comparisons = []

        # Run each config
        for cfg_def in scenario_def['configs']:
            name = cfg_def['name']
            print(f"\nRunning config: {name}")

            scalars = {m: [] for m in SCALAR_METRICS}
            ts = {m: [] for m in TS_METRICS}

            for seed_idx, seed in enumerate(self.seeds):
                print(f"Seed {seed_idx+1}/{self.seeds}", end='\r')
                run_cfg = {k: v for k, v in cfg_def.items() if k != 'name'}
                metrics, detailed = run_simulation(seed, run_cfg)

                for m in SCALAR_METRICS:
                    scalars[m].append(metrics.get(m, np.nan))

                for m in TS_METRICS:
                    if m in detailed and isinstance(detailed[m], list) and len(detailed[m]) > 0:
                        try:
                            arr = np.array(detailed[m], dtype=float)
                            ts[m].append(arr)
                        except:
                            ts[m].append(np.array([]))
                    else:
                        ts[m].append(np.array([]))
            print()

            for m in SCALAR_METRICS:
                scalars[m] = np.array(scalars[m], dtype=float)

            scenario_results[name] = {'scalars': scalars, 'ts': ts}

            # Statistical comparisons (only if enough valid samples)
            valid_metrics = 0
            for metric in SCALAR_METRICS:
                x = baseline_scalars[metric]
                y = scalars[metric]

                mask = np.isfinite(x) & np.isfinite(y)
                x_p, y_p = x[mask], y[mask]

                if len(x_p) < 5:
                    continue  # Skip metrics with insufficient samples

                valid_metrics += 1

                # Wilcoxon test
                try:
                    stat, p_raw = wilcoxon(x_p, y_p)
                except Exception as e:
                    print(f"Wilcoxon failed for {metric}: {e}")
                    continue

                # Effect size & CIs
                d = paired_cohens_d(x, y)
                mean_b, ci_b_low, ci_b_high = ci95_t(x)
                mean_c, ci_c_low, ci_c_high = ci95_t(y)

                all_comparisons.append({
                    'scenario': scenario_name,
                    'metric': metric,
                    'baseline': baseline_name,
                    'config': name,
                    'mean_baseline': mean_b,
                    'ci95_baseline_low': ci_b_low,
                    'ci95_baseline_high': ci_b_high,
                    'mean_config': mean_c,
                    'ci95_config_low': ci_c_low,
                    'ci95_config_high': ci_c_high,
                    'diff': mean_c - mean_b,
                    'p_raw': p_raw,
                    'cohens_d': d,
                    'significant_raw': p_raw < 0.05
                })

            if valid_metrics == 0:
                print(
                    f"WARNING: No metrics had ≥5 valid samples for config '{name}'")
                print(f"Check simulation stability (early network death?)")

        # Save results ONLY if comparisons exist
        if all_comparisons:
            # FDR correction
            p_vals = [comp['p_raw'] for comp in all_comparisons]
            reject, p_adj, _, _ = multipletests(
                p_vals, alpha=0.05, method='fdr_bh')

            baseline_rows = []
            for metric in SCALAR_METRICS:
                x = baseline_scalars[metric]
                mean_b, ci_b_low, ci_b_high = ci95_t(x)
                # Only add if metric has valid data
                if not np.isnan(mean_b):
                    baseline_rows.append({
                        'scenario': scenario_name,
                        'metric': metric,
                        'baseline': baseline_name,
                        'config': baseline_name,  # Mark as baseline itself
                        'mean_baseline': mean_b,
                        'ci95_baseline_low': ci_b_low,
                        'ci95_baseline_high': ci_b_high,
                        'mean_config': mean_b,
                        'ci95_config_low': ci_b_low,
                        'ci95_config_high': ci_b_high,
                        'diff': 0.0,
                        'p_raw': np.nan,
                        'cohens_d': 0.0,
                        'significant_raw': False,
                        'p_adj': np.nan,
                        'significant_fdr': False
                    })

            # PREPEND baseline rows to comparisons (appears first in CSV/JSON)
            all_comparisons = baseline_rows + all_comparisons

            for i, comp in enumerate(all_comparisons):
                comp['p_adj'] = p_adj[i] if i < len(p_adj) else np.nan
                comp['significant_fdr'] = reject[i] if i < len(
                    reject) else False

            df = pd.DataFrame(all_comparisons)
            df.to_csv(results_dir / "statistical_summary.csv", index=False)
            df.to_json(results_dir / "statistical_summary.json",
                       orient='records', indent=2)
            print(f"Saved {len(all_comparisons)} comparisons to {results_dir}")
        else:
            print(f"NO VALID COMPARISONS for scenario '{scenario_name}'")
            print(f"Results NOT saved (empty file would cause errors)")
            # Create empty placeholder file to avoid missing file errors later
            with open(results_dir / "statistical_summary.csv", 'w') as f:
                f.write("scenario,metric,baseline,config,mean_baseline,ci95_baseline_low,ci95_baseline_high,mean_config,ci95_config_low,ci95_config_high,diff,p_raw,cohens_d,significant_raw,p_adj,significant_fdr\n")

        # Generate plots (skip if no data)
        try:
            self._generate_plots(scenario_name, baseline_name, baseline_scalars, baseline_ts,
                                 scenario_results, scenario_def, plots_dir)
        except Exception as e:
            print(f"Plot generation failed (skipping): {e}")

        return all_comparisons

    def _generate_plots(self, scenario_name: str, baseline_name: str,
                        baseline_scalars: Dict[str, np.ndarray],
                        baseline_ts: Dict[str, List[np.ndarray]],
                        scenario_results: Dict[str, Dict],
                        scenario_def: Dict[str, Any],
                        output_dir: Path):
        """Generate plots with robust empty-data handling."""
        config_names = [baseline_name] + list(scenario_results.keys())
        colors = sns.color_palette("husl", n_colors=len(config_names))

        # Boxplots for key metrics
        key_metrics = ["FND", "HND", "LND", "PDR", "TotalDelivered", "TotalGenerated",
                       "Avg_E2E_Delay_Rounds"]
        for metric in key_metrics:
            plt.figure(figsize=(8, 6))
            data = []
            labels = []

            # Baseline
            vals = baseline_scalars[metric][np.isfinite(
                baseline_scalars[metric])]
            if len(vals) > 0:
                data.append(vals)
                labels.append(baseline_name)

            # Configs
            for idx, (name, res) in enumerate(scenario_results.items()):
                vals = res['scalars'][metric][np.isfinite(
                    res['scalars'][metric])]
                if len(vals) > 0:
                    data.append(vals)
                    labels.append(name)

            if not data:
                plt.close()
                continue

            bp = plt.boxplot(data, patch_artist=True, labels=labels, showmeans=True,
                             meanprops=dict(marker='D', markeredgecolor='black', markerfacecolor='white'))

            for patch, color in zip(bp['boxes'], colors[:len(data)]):
                patch.set_facecolor(color)
                patch.set_alpha(0.7)

            plt.title(f"{scenario_name.replace('_', ' ').title()}: {metric}")
            plt.ylabel(metric)
            plt.grid(axis='y', linestyle='--', alpha=0.7)
            plt.xticks(rotation=15, ha='right')
            plt.tight_layout()
            plt.savefig(output_dir / f"boxplot_{metric}.png", dpi=300)
            plt.close()

        # Time-series for EC (only if data exists)
        if 'EC' in baseline_ts and any(len(arr) > 0 for arr in baseline_ts['EC']):
            plt.figure(figsize=(10, 6))

            # Helper to plot a config's EC
            def plot_ec(name, ec_list, color, label):
                valid_arrs = [a for a in ec_list if len(a) > 0]
                if not valid_arrs:
                    return
                max_len = max(len(a) for a in valid_arrs)
                mat = np.full((len(valid_arrs), max_len), np.nan)
                for i, a in enumerate(valid_arrs):
                    mat[i, :len(a)] = a[:max_len]
                mean_vals = np.nanmean(mat, axis=0)
                n_vals = np.sum(~np.isnan(mat), axis=0)
                sd_vals = np.nanstd(mat, axis=0, ddof=1)

                # Create a mask for valid indices (n >= 2)
                valid_mask = n_vals >= 2
                ci_vals = np.zeros_like(mean_vals)
                # Vectorized calculation only on valid indices
                se = sd_vals[valid_mask] / np.sqrt(n_vals[valid_mask])
                t_crit = stats.t.ppf(0.975, n_vals[valid_mask] - 1)
                ci_vals[valid_mask] = se * t_crit

                x = np.arange(len(mean_vals))
                plt.plot(x, mean_vals[:len(x)],
                         label=label, color=color, linewidth=2)
                plt.fill_between(x,
                                 (mean_vals - ci_vals)[:len(x)],
                                 (mean_vals + ci_vals)[:len(x)],
                                 color=color, alpha=0.2)

            # Plot baseline
            plot_ec(baseline_name, baseline_ts['EC'], colors[0], baseline_name)

            # Plot configs
            for idx, (name, res) in enumerate(scenario_results.items()):
                if 'EC' in res['ts']:
                    plot_ec(name, res['ts']['EC'], colors[idx+1], name)

            plt.xlabel("Round index (logged every 50 rounds)")
            plt.ylabel("Energy Consumption (Joules)")
            plt.title(
                f"Energy Consumption: {scenario_name.replace('_', ' ').title()}")
            plt.legend()
            plt.grid(True, linestyle='--', alpha=0.6)
            plt.tight_layout()
            plt.savefig(output_dir / "timeseries_EC.png", dpi=300)
            plt.close()

# MAIN EXECUTION


def main(seeds):
    print("EFFICIENT WSN EVALUATION FRAMEWORK (FIXED FOR EMPTY RESULTS)")
    print("Baseline run ONCE → reused across scenarios with robust error handling")

    evaluator = EfficientEvaluator(seeds=seeds)
    evaluator.run_baseline_once()

    # Run scenarios
    scenario_results = {}
    for scenario_name, scenario_def in SCENARIOS.items():
        try:
            print(f"\n{'>'*70}")
            comps = evaluator.run_scenario(scenario_name, scenario_def)
            scenario_results[scenario_name] = comps
            print(f"{'<'*70}")
        except Exception as e:
            print(f"\nCRITICAL ERROR in scenario '{scenario_name}': {e}")
            import traceback
            traceback.print_exc()
            continue

    print("\n")
    print("EVALUATION COMPLETE")
    print(f"Results: {RESULTS_DIR.absolute()}")
    print(f"Plots:   {PLOTS_DIR.absolute()}")

    # Generate master summary (skip empty files)
    master_rows = []
    for scenario_name in SCENARIOS:
        path = RESULTS_DIR / scenario_name / "statistical_summary.csv"
        if path.exists():
            try:
                df = pd.read_csv(path)
                if len(df) > 0:  # Only include non-empty results
                    master_rows.append(df)
                else:
                    print(
                        f"Skipping empty results for scenario '{scenario_name}'")
            except pd.errors.EmptyDataError:
                print(
                    f"Empty CSV file for scenario '{scenario_name}' - skipping")
            except Exception as e:
                print(f"Error reading {scenario_name} results: {e}")

    if master_rows:
        master_df = pd.concat(master_rows, ignore_index=True)
        master_df.to_csv(RESULTS_DIR / "master_summary.csv", index=False)
        print(f"\n Master summary saved: {RESULTS_DIR / 'master_summary.csv'}")
        print(
            f"Contains {len(master_df)} valid comparisons across {master_df['scenario'].nunique()} scenarios")
    else:
        print("\nNO VALID COMPARISONS ACROSS ALL SCENARIOS")
        print("Check simulation stability - likely early network death causing NaNs")


if __name__ == "__main__":
    SEEDS = [
        1, 3, 4, 18, 29, 31, 41, 53, 61, 67,
        73, 83, 89, 97, 107, 109, 137, 139, 163, 173,
        179, 181, 191, 197, 227, 229, 233, 239, 241, 251]
    main(seeds=SEEDS)

## Random Node Deployment vs. Voronoi-RL 

### (100, 100) area size, 100 nodes

In [ ]:
# Run both methods
drl_lifetimes, plot_drl = run_multiple_simulations(area_size=(
    100, 100), n_nodes=100, sink_mode="eeosp",
    routing_mode="multi-hop", mode="random", n_runs=30, tune_edge_iterations=50)

baseline_lifetimes, plot_base = run_multiple_simulations(area_size=(
    100, 100), n_nodes=100, sink_mode="eeosp", routing_mode="multi-hop", mode="random", n_runs=30)


print(
    f"Mean DRL FND: {drl_lifetimes[0].mean():.2f} ± {drl_lifetimes[0].std():.2f}")
print(
    f"Mean DRL HND: {drl_lifetimes[1].mean():.2f} ± {drl_lifetimes[1].std():.2f}")
print(
    f"Mean DRL LND: {drl_lifetimes[2].mean():.2f} ± {drl_lifetimes[2].std():.2f}")
print(
    f"Mean DRL PDR: {drl_lifetimes[3].mean():.2f} ± {drl_lifetimes[3].std():.2f}")
print(
    f"Mean DRL TotalGenerated: {drl_lifetimes[4].mean():.2f} ± {drl_lifetimes[4].std():.2f}")
print(
    f"Mean DRL TotalDelivered: {drl_lifetimes[5].mean():.2f} ± {drl_lifetimes[5].std():.2f}")
print(
    f"Mean DRL Avg_E2E_Delay_Rounds: {drl_lifetimes[6].mean():.2f} ± {drl_lifetimes[6].std():.2f}")
print(
    f"Mean DRL Avg_E2E_Delay_Sec: {drl_lifetimes[7].mean():.2f} ± {drl_lifetimes[7].std():.2f}")


print("\n\n\n")
print(
    f"Mean Baseline FND: {baseline_lifetimes[0].mean():.2f} ± {baseline_lifetimes[0].std():.2f}")
print(
    f"Mean Baseline HND: {baseline_lifetimes[1].mean():.2f} ± {baseline_lifetimes[1].std():.2f}")
print(
    f"Mean Baseline LND: {baseline_lifetimes[2].mean():.2f} ± {baseline_lifetimes[2].std():.2f}")
print(
    f"Mean Baseline PDR: {baseline_lifetimes[3].mean():.2f} ± {baseline_lifetimes[3].std():.2f}")
print(
    f"Mean Baseline TotalGenerated: {baseline_lifetimes[4].mean():.2f} ± {baseline_lifetimes[4].std():.2f}")
print(
    f"Mean Baseline TotalDelivered: {baseline_lifetimes[5].mean():.2f} ± {baseline_lifetimes[5].std():.2f}")
print(
    f"Mean Baseline Avg_E2E_Delay_Rounds: {baseline_lifetimes[6].mean():.2f} ± {baseline_lifetimes[6].std():.2f}")
print(
    f"Mean Baseline Avg_E2E_Delay_Sec: {baseline_lifetimes[7].mean():.2f} ± {baseline_lifetimes[7].std():.2f}")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(20, 20))

# EC
plt.subplot(4, 4, 1)
plot_time_series(plot_drl, 'EC', "Voronoi-RL", color='green')
plot_time_series(plot_base, 'EC', "Random", color='purple')
plt.xlabel('Round')
plt.ylabel('Coverage (EC)')
plt.title('Time')
plt.grid(True)
plt.legend()

# avg_RE
plt.subplot(4, 4, 2)
plot_time_series(plot_drl, 'avg_RE', "Voronoi-RL", color='green')
plot_time_series(plot_base, 'avg_RE', "Random", color='purple')
plt.xlabel('Round')
plt.ylabel('Load Balance (avg_RE)')
plt.title('Time')
plt.grid(True)
plt.legend()

# TH
plt.subplot(4, 4, 3)
plot_time_series(plot_drl, 'TH', "Voronoi-RL", color='green')
plot_time_series(plot_base, 'TH', "Random", color='purple')
plt.xlabel('Round')
plt.ylabel('TH')
plt.title('TH')
plt.grid(True)
plt.legend()

# TH_pps
plt.subplot(4, 4, 4)
plot_time_series(plot_drl, 'TH_pps', "Voronoi-RL", color='green')
plot_time_series(plot_base, 'TH_pps', "Random", color='purple')
plt.xlabel('Round')
plt.ylabel('THpps')
plt.title('TH_pps')
plt.grid(True)
plt.legend()

# PLR
plt.subplot(4, 4, 5)
plot_time_series(plot_drl, 'PLR', "Voronoi-RL", color='green')
plot_time_series(plot_base, 'PLR', "Random", color='purple')
plt.xlabel('Round')
plt.ylabel('PLR')
plt.title('PLR')
plt.grid(True)
plt.legend()

# LB
plt.subplot(4, 4, 6)
plot_time_series(plot_drl, 'LB', "Voronoi-RL", color='green')
plot_time_series(plot_base, 'LB', "Random", color='purple')
plt.xlabel('Round')
plt.ylabel('LB')
plt.title('LB')
plt.grid(True)
plt.legend()

# FI
plt.subplot(4, 4, 7)
plot_time_series(plot_drl, 'FI', "Voronoi-RL", color='green')
plot_time_series(plot_base, 'FI', "Random", color='purple')
plt.xlabel('Round')
plt.ylabel('FI')
plt.title('FI')
plt.grid(True)
plt.legend()

# CE
plt.subplot(4, 4, 8)
plot_time_series(plot_drl, 'CE', "Voronoi-RL", color='green')
plot_time_series(plot_base, 'CE', "Random", color='purple')
plt.xlabel('Round')
plt.ylabel('CE')
plt.title('CE')
plt.grid(True)
plt.legend()

# Buffer_Overflow_Rate
plt.subplot(4, 4, 9)
plot_time_series(plot_drl, 'Buffer_Overflow_Rate', "Voronoi-RL", color='green')
plot_time_series(plot_base, 'Buffer_Overflow_Rate', "Random", color='purple')
plt.xlabel('Round')
plt.ylabel('Bufer_Overflow_Rate')
plt.title('Buffer_Overflow_Rate')
plt.grid(True)
plt.legend()

# CA
plt.subplot(4, 4, 10)
plot_time_series(plot_drl, 'CA', "Voronoi-RL", color='green')
plot_time_series(plot_base, 'CA', "Random", color='purple')
plt.xlabel('Round')
plt.ylabel('CA')
plt.title('CA')
plt.grid(True)
plt.legend()

# RL
plt.subplot(4, 4, 11)
plot_time_series(plot_drl, 'RL', "Voronoi-RL", color='green')
plot_time_series(plot_base, 'RL', "Random", color='purple')
plt.xlabel('Round')
plt.ylabel('RL')
plt.title('RL')
plt.grid(True)
plt.legend()

# EE
plt.subplot(4, 4, 12)
plot_time_series(plot_drl, 'EE', "Voronoi-RL", color='green')
plot_time_series(plot_base, 'EE', "Random", color='purple')
plt.xlabel('Round')
plt.ylabel('EE')
plt.title('EE')
plt.grid(True)
plt.legend()

# EE_Js
plt.subplot(4, 4, 13)
plot_time_series(plot_drl, 'EE_Js', "Voronoi-RL", color='green')
plot_time_series(plot_base, 'EE_Js', "Random", color='purple')
plt.xlabel('Round')
plt.ylabel('EEJs')
plt.title('EE_Js')
plt.grid(True)
plt.legend()

# routing_Overhead_Bytes
plt.subplot(4, 4, 14)
plot_time_series(plot_drl, 'routing_Overhead_Bytes',
                 "Voronoi-RL", color='green')
plot_time_series(plot_base, 'routing_Overhead_Bytes', "Random", color='purple')
plt.xlabel('Round')
plt.ylabel('roting_Overhead_Bytes')
plt.title('routing_Overhead_Bytes')
plt.grid(True)
plt.legend()

# Traffic_Load_Pct
plt.subplot(4, 4, 15)
plot_time_series(plot_drl, 'Traffic_Load_Pct', "Voronoi-RL", color='green')
plot_time_series(plot_base, 'Traffic_Load_Pct', "Random", color='purple')
plt.xlabel('Round')
plt.ylabel('Trffic_Load_Pct')
plt.title('Traffic_Load_Pct')
plt.grid(True)
plt.legend()

# Overhead_Normalized
plt.subplot(4, 4, 16)
plot_time_series(plot_drl, 'Overhead_Normalized', "Voronoi-RL", color='green')
plot_time_series(plot_base, 'Overhead_Normalized', "Random", color='purple')
plt.xlabel('Round')
plt.ylabel('Overhead_Normalized')
plt.title('Overhead_Normalized')
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.savefig('metrics_over_time.png', dpi=300, bbox_inches='tight')
plt.show()

### (200, 200) area size, 100 nodes

In [ ]:
# Run both methods
drl_lifetimes, plot_drl = run_multiple_simulations(area_size=(
    200, 200), n_nodes=100, sink_mode="eeosp", routing_mode="multi-hop", mode="DRL", n_runs=30)
baseline_lifetimes, plot_base = run_multiple_simulations(area_size=(
    200, 200), n_nodes=100, sink_mode="eeosp", routing_mode="multi-hop", mode="random", n_runs=30)


print(
    f"Mean DRL FND: {drl_lifetimes[0].mean():.2f} ± {drl_lifetimes[0].std():.2f}")
print(
    f"Mean DRL HND: {drl_lifetimes[1].mean():.2f} ± {drl_lifetimes[1].std():.2f}")
print(
    f"Mean DRL LND: {drl_lifetimes[2].mean():.2f} ± {drl_lifetimes[2].std():.2f}")
print(
    f"Mean DRL PDR: {drl_lifetimes[3].mean():.2f} ± {drl_lifetimes[3].std():.2f}")
print(
    f"Mean DRL TotalGenerated: {drl_lifetimes[4].mean():.2f} ± {drl_lifetimes[4].std():.2f}")
print(
    f"Mean DRL TotalDelivered: {drl_lifetimes[5].mean():.2f} ± {drl_lifetimes[5].std():.2f}")
print(
    f"Mean DRL Avg_E2E_Delay_Rounds: {drl_lifetimes[6].mean():.2f} ± {drl_lifetimes[6].std():.2f}")
print(
    f"Mean DRL Avg_E2E_Delay_Sec: {drl_lifetimes[7].mean():.2f} ± {drl_lifetimes[7].std():.2f}")


print("\n\n\n")
print(
    f"Mean Baseline FND: {baseline_lifetimes[0].mean():.2f} ± {baseline_lifetimes[0].std():.2f}")
print(
    f"Mean Baseline HND: {baseline_lifetimes[1].mean():.2f} ± {baseline_lifetimes[1].std():.2f}")
print(
    f"Mean Baseline LND: {baseline_lifetimes[2].mean():.2f} ± {baseline_lifetimes[2].std():.2f}")
print(
    f"Mean Baseline PDR: {baseline_lifetimes[3].mean():.2f} ± {baseline_lifetimes[3].std():.2f}")
print(
    f"Mean Baseline TotalGenerated: {baseline_lifetimes[4].mean():.2f} ± {baseline_lifetimes[4].std():.2f}")
print(
    f"Mean Baseline TotalDelivered: {baseline_lifetimes[5].mean():.2f} ± {baseline_lifetimes[5].std():.2f}")
print(
    f"Mean Baseline Avg_E2E_Delay_Rounds: {baseline_lifetimes[6].mean():.2f} ± {baseline_lifetimes[6].std():.2f}")
print(
    f"Mean Baseline Avg_E2E_Delay_Sec: {baseline_lifetimes[7].mean():.2f} ± {baseline_lifetimes[7].std():.2f}")

In [ ]:
plt.figure(figsize=(15, 5))

# CA
plt.subplot(1, 3, 1)
plot_time_series(plot_drl, 'CA', "Voronoi-RL", color='green')
plot_time_series(plot_base, 'CA', "Random", color='purple')
plt.xlabel('Round')
plt.ylabel('Coverage (CA)')
plt.title('Spatial Coverage over Time')
plt.grid(True)
plt.legend()

# LB
plt.subplot(1, 3, 2)
plot_time_series(plot_drl, 'LB', "Voronoi-RL", color='green')
plot_time_series(plot_base, 'LB', "Random", color='purple')
plt.xlabel('Round')
plt.ylabel('Load Balance (LB)')
plt.title('Load Balance over Time')
plt.grid(True)
plt.legend()

# CE
plt.subplot(1, 3, 3)
plot_time_series(plot_drl, 'EC', "Voronoi-RL", color='green')
plot_time_series(plot_base, 'EC', "Random", color='purple')
plt.xlabel('Round')
plt.ylabel('Coverage Efficiency (EC)')
plt.title('Coverage per Energy Unit (EC)')
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.savefig('metrics_over_time.png', dpi=300, bbox_inches='tight')
plt.show()

### (500, 500) area size, 100 nodes

In [ ]:
# Run both methods
drl_lifetimes, plot_drl = run_multiple_simulations(area_size=(
    500, 500), n_nodes=100, sink_mode="eeosp", routing_mode="multi-hop", mode="DRL", n_runs=30, edge_threshold=0.01, tune_edge_iterations=10)
baseline_lifetimes, plot_base = run_multiple_simulations(area_size=(
    500, 500), n_nodes=100, sink_mode="eeosp", routing_mode="multi-hop", mode="random", n_runs=30)


print(
    f"Mean DRL FND: {drl_lifetimes[0].mean():.2f} ± {drl_lifetimes[0].std():.2f}")
print(
    f"Mean DRL HND: {drl_lifetimes[1].mean():.2f} ± {drl_lifetimes[1].std():.2f}")
print(
    f"Mean DRL LND: {drl_lifetimes[2].mean():.2f} ± {drl_lifetimes[2].std():.2f}")
print(
    f"Mean DRL PDR: {drl_lifetimes[3].mean():.2f} ± {drl_lifetimes[3].std():.2f}")
print(
    f"Mean DRL TotalGenerated: {drl_lifetimes[4].mean():.2f} ± {drl_lifetimes[4].std():.2f}")
print(
    f"Mean DRL TotalDelivered: {drl_lifetimes[5].mean():.2f} ± {drl_lifetimes[5].std():.2f}")
print(
    f"Mean DRL Avg_E2E_Delay_Rounds: {drl_lifetimes[6].mean():.2f} ± {drl_lifetimes[6].std():.2f}")
print(
    f"Mean DRL Avg_E2E_Delay_Sec: {drl_lifetimes[7].mean():.2f} ± {drl_lifetimes[7].std():.2f}")


print("\n\n\n")
print(
    f"Mean Baseline FND: {baseline_lifetimes[0].mean():.2f} ± {baseline_lifetimes[0].std():.2f}")
print(
    f"Mean Baseline HND: {baseline_lifetimes[1].mean():.2f} ± {baseline_lifetimes[1].std():.2f}")
print(
    f"Mean Baseline LND: {baseline_lifetimes[2].mean():.2f} ± {baseline_lifetimes[2].std():.2f}")
print(
    f"Mean Baseline PDR: {baseline_lifetimes[3].mean():.2f} ± {baseline_lifetimes[3].std():.2f}")
print(
    f"Mean Baseline TotalGenerated: {baseline_lifetimes[4].mean():.2f} ± {baseline_lifetimes[4].std():.2f}")
print(
    f"Mean Baseline TotalDelivered: {baseline_lifetimes[5].mean():.2f} ± {baseline_lifetimes[5].std():.2f}")
print(
    f"Mean Baseline Avg_E2E_Delay_Rounds: {baseline_lifetimes[6].mean():.2f} ± {baseline_lifetimes[6].std():.2f}")
print(
    f"Mean Baseline Avg_E2E_Delay_Sec: {baseline_lifetimes[7].mean():.2f} ± {baseline_lifetimes[7].std():.2f}")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(15, 5))

# CA
plt.subplot(1, 3, 1)
plot_time_series(plot_drl, 'CA', "Voronoi-RL", color='green')
plot_time_series(plot_base, 'CA', "Random", color='purple')
plt.xlabel('Round')
plt.ylabel('Coverage (CA)')
plt.title('Spatial Coverage over Time')
plt.grid(True)
plt.legend()

# LB
plt.subplot(1, 3, 2)
plot_time_series(plot_drl, 'LB', "Voronoi-RL", color='green')
plot_time_series(plot_base, 'LB', "Random", color='purple')
plt.xlabel('Round')
plt.ylabel('Load Balance (LB)')
plt.title('Load Balance over Time')
plt.grid(True)
plt.legend()

# CE
plt.subplot(1, 3, 3)
plot_time_series(plot_drl, 'EC', "Voronoi-RL", color='green')
plot_time_series(plot_base, 'EC', "Random", color='purple')
plt.xlabel('Round')
plt.ylabel('Coverage Efficiency (EC)')
plt.title('Coverage per Energy Unit (EC)')
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.savefig('metrics_over_time.png', dpi=300, bbox_inches='tight')
plt.show()

### (1000, 1000) area size, 100 nodes

In [ ]:
# Run both methods
drl_lifetimes, plot_drl = run_multiple_simulations(area_size=(
    1000, 1000), n_nodes=100, sink_mode="eeosp", routing_mode="multi-hop", mode="DRL", n_runs=30, edge_threshold=0.01, tune_edge_iterations=10)
baseline_lifetimes, plot_base = run_multiple_simulations(area_size=(
    1000, 1000), n_nodes=100, sink_mode="eeosp", routing_mode="multi-hop", mode="random", n_runs=30)


print(
    f"Mean DRL FND: {drl_lifetimes[0].mean():.2f} ± {drl_lifetimes[0].std():.2f}")
print(
    f"Mean DRL HND: {drl_lifetimes[1].mean():.2f} ± {drl_lifetimes[1].std():.2f}")
print(
    f"Mean DRL LND: {drl_lifetimes[2].mean():.2f} ± {drl_lifetimes[2].std():.2f}")
print(
    f"Mean DRL PDR: {drl_lifetimes[3].mean():.2f} ± {drl_lifetimes[3].std():.2f}")
print(
    f"Mean DRL TotalGenerated: {drl_lifetimes[4].mean():.2f} ± {drl_lifetimes[4].std():.2f}")
print(
    f"Mean DRL TotalDelivered: {drl_lifetimes[5].mean():.2f} ± {drl_lifetimes[5].std():.2f}")
print(
    f"Mean DRL Avg_E2E_Delay_Rounds: {drl_lifetimes[6].mean():.2f} ± {drl_lifetimes[6].std():.2f}")
print(
    f"Mean DRL Avg_E2E_Delay_Sec: {drl_lifetimes[7].mean():.2f} ± {drl_lifetimes[7].std():.2f}")


print("\n\n\n")
print(
    f"Mean Baseline FND: {baseline_lifetimes[0].mean():.2f} ± {baseline_lifetimes[0].std():.2f}")
print(
    f"Mean Baseline HND: {baseline_lifetimes[1].mean():.2f} ± {baseline_lifetimes[1].std():.2f}")
print(
    f"Mean Baseline LND: {baseline_lifetimes[2].mean():.2f} ± {baseline_lifetimes[2].std():.2f}")
print(
    f"Mean Baseline PDR: {baseline_lifetimes[3].mean():.2f} ± {baseline_lifetimes[3].std():.2f}")
print(
    f"Mean Baseline TotalGenerated: {baseline_lifetimes[4].mean():.2f} ± {baseline_lifetimes[4].std():.2f}")
print(
    f"Mean Baseline TotalDelivered: {baseline_lifetimes[5].mean():.2f} ± {baseline_lifetimes[5].std():.2f}")
print(
    f"Mean Baseline Avg_E2E_Delay_Rounds: {baseline_lifetimes[6].mean():.2f} ± {baseline_lifetimes[6].std():.2f}")
print(
    f"Mean Baseline Avg_E2E_Delay_Sec: {baseline_lifetimes[7].mean():.2f} ± {baseline_lifetimes[7].std():.2f}")

In [ ]:
plt.figure(figsize=(15, 5))

# CA
plt.subplot(1, 3, 1)
plot_time_series(plot_drl, 'CA', "Voronoi-RL", color='green')
plot_time_series(plot_base, 'CA', "Random", color='purple')
plt.xlabel('Round')
plt.ylabel('Coverage (CA)')
plt.title('Spatial Coverage over Time')
plt.grid(True)
plt.legend()

# LB
plt.subplot(1, 3, 2)
plot_time_series(plot_drl, 'LB', "Voronoi-RL", color='green')
plot_time_series(plot_base, 'LB', "Random", color='purple')
plt.xlabel('Round')
plt.ylabel('Load Balance (LB)')
plt.title('Load Balance over Time')
plt.grid(True)
plt.legend()

# CE
plt.subplot(1, 3, 3)
plot_time_series(plot_drl, 'EC', "Voronoi-RL", color='green')
plot_time_series(plot_base, 'EC', "Random", color='purple')
plt.xlabel('Round')
plt.ylabel('Coverage Efficiency (EC)')
plt.title('Coverage per Energy Unit (EC)')
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.savefig('metrics_over_time.png', dpi=300, bbox_inches='tight')
plt.show()

## Different MS patterns

In [ ]:
# Run both methods
eeosp_lifetimes, plot_eeosp = run_multiple_simulations(area_size=(
    100, 100), n_nodes=100, sink_mode="eeosp", routing_mode="multi-hop", mode="random", n_runs=30)
adaptive_lifetimes, plot_adaptive = run_multiple_simulations(area_size=(
    100, 100), n_nodes=100, sink_mode="adaptive", routing_mode="multi-hop", mode="random", n_runs=30)
fixed_lifetimes, plot_fixed = run_multiple_simulations(area_size=(
    100, 100), n_nodes=100, sink_mode="fixed", routing_mode="multi-hop", mode="random", n_runs=30)
random_lifetimes, plot_random = run_multiple_simulations(area_size=(
    100, 100), n_nodes=100, sink_mode="random", routing_mode="multi-hop", mode="random", n_runs=30)

print(
    f"Mean eeosp FND: {eeosp_lifetimes[0].mean():.2f} ± {eeosp_lifetimes[0].std():.2f}")
print(
    f"Mean eeosp HND: {eeosp_lifetimes[1].mean():.2f} ± {eeosp_lifetimes[1].std():.2f}")
print(
    f"Mean eeosp LND: {eeosp_lifetimes[2].mean():.2f} ± {eeosp_lifetimes[2].std():.2f}")
print(
    f"Mean eeosp PDR: {eeosp_lifetimes[3].mean():.2f} ± {eeosp_lifetimes[3].std():.2f}")
print(
    f"Mean eeosp TotalGenerated: {eeosp_lifetimes[4].mean():.2f} ± {eeosp_lifetimes[4].std():.2f}")
print(
    f"Mean eeosp TotalDelivered: {eeosp_lifetimes[5].mean():.2f} ± {eeosp_lifetimes[5].std():.2f}")
print(
    f"Mean eeosp Avg_E2E_Delay_Rounds: {eeosp_lifetimes[6].mean():.2f} ± {eeosp_lifetimes[6].std():.2f}")
print(
    f"Mean eeosp Avg_E2E_Delay_Sec: {eeosp_lifetimes[7].mean():.2f} ± {eeosp_lifetimes[7].std():.2f}")


print("\n\n")
print(
    f"Mean adaptive FND: {adaptive_lifetimes[0].mean():.2f} ± {adaptive_lifetimes[0].std():.2f}")
print(
    f"Mean adaptive HND: {adaptive_lifetimes[1].mean():.2f} ± {adaptive_lifetimes[1].std():.2f}")
print(
    f"Mean adaptive LND: {adaptive_lifetimes[2].mean():.2f} ± {adaptive_lifetimes[2].std():.2f}")
print(
    f"Mean adaptive PDR: {adaptive_lifetimes[3].mean():.2f} ± {adaptive_lifetimes[3].std():.2f}")
print(
    f"Mean adaptive TotalGenerated: {adaptive_lifetimes[4].mean():.2f} ± {adaptive_lifetimes[4].std():.2f}")
print(
    f"Mean adaptive TotalDelivered: {adaptive_lifetimes[5].mean():.2f} ± {adaptive_lifetimes[5].std():.2f}")
print(
    f"Mean adaptive Avg_E2E_Delay_Rounds: {adaptive_lifetimes[6].mean():.2f} ± {adaptive_lifetimes[6].std():.2f}")
print(
    f"Mean adaptive Avg_E2E_Delay_Sec: {adaptive_lifetimes[7].mean():.2f} ± {adaptive_lifetimes[7].std():.2f}")


print("\n\n")
print(
    f"Mean fixed FND: {fixed_lifetimes[0].mean():.2f} ± {fixed_lifetimes[0].std():.2f}")
print(
    f"Mean fixed HND: {fixed_lifetimes[1].mean():.2f} ± {fixed_lifetimes[1].std():.2f}")
print(
    f"Mean fixed LND: {fixed_lifetimes[2].mean():.2f} ± {fixed_lifetimes[2].std():.2f}")
print(
    f"Mean fixed PDR: {fixed_lifetimes[3].mean():.2f} ± {fixed_lifetimes[3].std():.2f}")
print(
    f"Mean fixed TotalGenerated: {fixed_lifetimes[4].mean():.2f} ± {fixed_lifetimes[4].std():.2f}")
print(
    f"Mean fixed TotalDelivered: {fixed_lifetimes[5].mean():.2f} ± {fixed_lifetimes[5].std():.2f}")
print(
    f"Mean fixed Avg_E2E_Delay_Rounds: {fixed_lifetimes[6].mean():.2f} ± {fixed_lifetimes[6].std():.2f}")
print(
    f"Mean fixed Avg_E2E_Delay_Sec: {fixed_lifetimes[7].mean():.2f} ± {fixed_lifetimes[7].std():.2f}")


print("\n\n")
print(
    f"Mean random FND: {random_lifetimes[0].mean():.2f} ± {random_lifetimes[0].std():.2f}")
print(
    f"Mean random HND: {random_lifetimes[1].mean():.2f} ± {random_lifetimes[1].std():.2f}")
print(
    f"Mean random LND: {random_lifetimes[2].mean():.2f} ± {random_lifetimes[2].std():.2f}")
print(
    f"Mean random PDR: {random_lifetimes[3].mean():.2f} ± {random_lifetimes[3].std():.2f}")
print(
    f"Mean random TotalGenerated: {random_lifetimes[4].mean():.2f} ± {random_lifetimes[4].std():.2f}")
print(
    f"Mean random TotalDelivered: {random_lifetimes[5].mean():.2f} ± {random_lifetimes[5].std():.2f}")
print(
    f"Mean random Avg_E2E_Delay_Rounds: {random_lifetimes[6].mean():.2f} ± {random_lifetimes[6].std():.2f}")
print(
    f"Mean random Avg_E2E_Delay_Sec: {random_lifetimes[7].mean():.2f} ± {random_lifetimes[7].std():.2f}")

In [ ]:
plt.figure(figsize=(15, 5))

# CA
plt.subplot(1, 3, 1)
plot_time_series(plot_eeosp, 'CA', "EEOSP", color='green')
plot_time_series(plot_adaptive, 'CA', "adaptive", color='purple')
plot_time_series(plot_fixed, 'CA', "fixed", color='blue')
plot_time_series(plot_random, 'CA', "random", color='red')
plt.xlabel('Round')
plt.ylabel('Coverage (CA)')
plt.title('Spatial Coverage over Time')
plt.grid(True)
plt.legend()

# LB
plt.subplot(1, 3, 2)
plot_time_series(plot_eeosp, 'LB', "EEOSP", color='green')
plot_time_series(plot_adaptive, 'LB', "adaptive", color='purple')
plot_time_series(plot_fixed, 'LB', "fixed", color='blue')
plot_time_series(plot_random, 'LB', "random", color='red')
plt.xlabel('Round')
plt.ylabel('Load Balance (LB)')
plt.title('Load Balance over Time')
plt.grid(True)
plt.legend()

# CE
plt.subplot(1, 3, 3)
plot_time_series(plot_eeosp, 'EC', "EEOSP", color='green')
plot_time_series(plot_adaptive, 'EC', "adaptive", color='purple')
plot_time_series(plot_fixed, 'EC', "fixed", color='blue')
plot_time_series(plot_random, 'EC', "random", color='red')
plt.xlabel('Round')
plt.ylabel('Coverage Efficiency (EC)')
plt.title('Coverage per Energy Unit (EC)')
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.savefig('metrics_over_time.png', dpi=300, bbox_inches='tight')
plt.show()

## Different clustering methods

In [ ]:
# Run both methods
OGSA_lifetimes, plot_OGSA = run_multiple_simulations(area_size=(
    100, 100), n_nodes=100, sink_mode="eeosp", routing_mode="multi-hop", mode="random", n_runs=30, CHS="optimizer")
adaptive_lifetimes, plot_adaptive = run_multiple_simulations(area_size=(
    100, 100), n_nodes=100, sink_mode="eeosp", routing_mode="multi-hop", mode="random", n_runs=30, CHS="adaptive")
# random_lifetimes, plot_random = run_multiple_simulations(area_size=(
    # 100, 100), n_nodes=100, sink_mode="eeosp", routing_mode="multi-hop", mode="random", n_runs=30, CHS="random")


print(
    f"Mean DRL FND: {OGSA_lifetimes[0].mean():.2f} ± {OGSA_lifetimes[0].std():.2f}")
print(
    f"Mean OGSA HND: {OGSA_lifetimes[1].mean():.2f} ± {OGSA_lifetimes[1].std():.2f}")
print(
    f"Mean OGSA LND: {OGSA_lifetimes[2].mean():.2f} ± {OGSA_lifetimes[2].std():.2f}")
print(
    f"Mean OGSA PDR: {OGSA_lifetimes[3].mean():.2f} ± {OGSA_lifetimes[3].std():.2f}")
print(
    f"Mean OGSA TotalGenerated: {OGSA_lifetimes[4].mean():.2f} ± {OGSA_lifetimes[4].std():.2f}")
print(
    f"Mean OGSA TotalDelivered: {OGSA_lifetimes[5].mean():.2f} ± {OGSA_lifetimes[5].std():.2f}")
print(
    f"Mean OGSA Avg_E2E_Delay_Rounds: {OGSA_lifetimes[6].mean():.2f} ± {OGSA_lifetimes[6].std():.2f}")
print(
    f"Mean OGSA Avg_E2E_Delay_Sec: {OGSA_lifetimes[7].mean():.2f} ± {OGSA_lifetimes[7].std():.2f}")


print("\n")
print(
    f"Mean adaptive FND: {adaptive_lifetimes[0].mean():.2f} ± {adaptive_lifetimes[0].std():.2f}")
print(
    f"Mean adaptive HND: {adaptive_lifetimes[1].mean():.2f} ± {adaptive_lifetimes[1].std():.2f}")
print(
    f"Mean adaptive LND: {adaptive_lifetimes[2].mean():.2f} ± {adaptive_lifetimes[2].std():.2f}")
print(
    f"Mean adaptive PDR: {adaptive_lifetimes[3].mean():.2f} ± {adaptive_lifetimes[3].std():.2f}")
print(
    f"Mean adaptive TotalGenerated: {adaptive_lifetimes[4].mean():.2f} ± {adaptive_lifetimes[4].std():.2f}")
print(
    f"Mean adaptive TotalDelivered: {adaptive_lifetimes[5].mean():.2f} ± {adaptive_lifetimes[5].std():.2f}")
print(
    f"Mean adaptive Avg_E2E_Delay_Rounds: {adaptive_lifetimes[6].mean():.2f} ± {adaptive_lifetimes[6].std():.2f}")
print(
    f"Mean adaptive Avg_E2E_Delay_Sec: {adaptive_lifetimes[7].mean():.2f} ± {adaptive_lifetimes[7].std():.2f}")


print("\n")
print(
    f"Mean random FND: {random_lifetimes[0].mean():.2f} ± {random_lifetimes[0].std():.2f}")
print(
    f"Mean random HND: {random_lifetimes[1].mean():.2f} ± {random_lifetimes[1].std():.2f}")
print(
    f"Mean random LND: {random_lifetimes[2].mean():.2f} ± {random_lifetimes[2].std():.2f}")
print(
    f"Mean random PDR: {random_lifetimes[3].mean():.2f} ± {random_lifetimes[3].std():.2f}")
print(
    f"Mean random TotalGenerated: {random_lifetimes[4].mean():.2f} ± {random_lifetimes[4].std():.2f}")
print(
    f"Mean random TotalDelivered: {random_lifetimes[5].mean():.2f} ± {random_lifetimes[5].std():.2f}")
print(
    f"Mean random Avg_E2E_Delay_Rounds: {random_lifetimes[6].mean():.2f} ± {random_lifetimes[6].std():.2f}")
print(
    f"Mean random Avg_E2E_Delay_Sec: {random_lifetimes[7].mean():.2f} ± {random_lifetimes[7].std():.2f}")

In [ ]:
plt.figure(figsize=(15, 5))

# CA
plt.subplot(1, 3, 1)
plot_time_series(plot_OGSA, 'CA', "OGSA", color='green')
plot_time_series(plot_adaptive, 'CA', "adaptive", color='purple')
plot_time_series(plot_random, 'CA', "random", color='red')
plt.xlabel('Round')
plt.ylabel('Coverage (CA)')
plt.title('Spatial Coverage over Time')
plt.grid(True)
plt.legend()

# LB
plt.subplot(1, 3, 2)
plot_time_series(plot_OGSA, 'LB', "OGSA", color='green')
plot_time_series(plot_adaptive, 'LB', "Random", color='purple')
plot_time_series(plot_random, 'LB', "random", color='red')
plt.xlabel('Round')
plt.ylabel('Load Balance (LB)')
plt.title('Load Balance over Time')
plt.grid(True)
plt.legend()

# CE
plt.subplot(1, 3, 3)
plot_time_series(plot_OGSA, 'EC', "OGSA", color='green')
plot_time_series(plot_adaptive, 'EC', "Random", color='purple')
plot_time_series(plot_random, 'EC', "random", color='red')
plt.xlabel('Round')
plt.ylabel('Coverage Efficiency (EC)')
plt.title('Coverage per Energy Unit (EC)')
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.savefig('metrics_over_time.png', dpi=300, bbox_inches='tight')
plt.show()